In [ ]:
from TrainingFramework import TrainingFramework
from ECGEncoder import ECGEncoder
from TextEncoder import TextEncoder
from SharedMetricSpace import SharedMetricSpace
from ContrastiveLearning import ContrastiveLearning
from ReportDecoder import ReportDecoder

from ECGDataLoader import ECGDataLoader, ECGDataBase
import warnings
warnings.filterwarnings("ignore")

TRAINING_PARAMS = \
{
    'num_epochs': 2,
    'batch_size': 8,
    'caption_loss_weight': 2,
    'contrastive_loss_weight': 1
}
#Load ECG images from MIMIC-IV-data-mini-subset/mimic-iv-ecg-matched-subset/mimic-iv-ecg_complete_300x300_images
ECG_DATA_DIR_MINI_SUBSET_IMAGES = "mimic-iv-ecg_complete_300x300_images"

{'num_epochs': 2, 'batch_size': 8, 'caption_loss_weight': 2, 'contrastive_loss_weight': 1}


In [ ]:
#Instantiate database
edb = ECGDataBase(ecg_data_dir=ECG_DATA_DIR_MINI_SUBSET_IMAGES)
#uncomment to randomly remove N-100 samples from the database:
#edb.random_undersample(100)

In [ ]:
#Subject-wise train val split 
tr_subjects, vl_subjects = edb.train_val_split()
print(f"tr_subjects: {len(tr_subjects)}")
print(f"vl_subjects: {len(vl_subjects)}")

In [ ]:
#Instantiate ECGDataLoader objects to get a torch.utils.data.DataLoaders for train and validation sets
tr_edl = ECGDataLoader(database=edb, split_subjects=tr_subjects, dynamic_loading=False, batch_size=TRAINING_PARAMS['batch_size'], shuffle=True)
tr_dl = tr_edl.get_dataloader()
vl_edl = ECGDataLoader(database=edb, split_subjects=vl_subjects, dynamic_loading=False, batch_size=TRAINING_PARAMS['batch_size'], shuffle=True)
vl_dl = vl_edl.get_dataloader()

In [ ]:
#Instantiate model components 
text_encoder = TextEncoder(pretrained_model="michiyasunaga/BioLinkBERT-base")
report_decoder = ReportDecoder(decoder_name = 'biogpt')
rd_emb_dim = report_decoder.decoder_input_dimension
ecg_encoder = ECGEncoder(model_architecture="ResNet101", representation_embedding_dim=rd_emb_dim, pretrained_model="resnetPTBXL_weights.pth")
te_emb_dim = text_encoder.embedding_dim
shared_metric_space = SharedMetricSpace(ecg_embedding_dim=ecg_encoder.representation_embedding_dim, text_embedding_dim=text_encoder.embedding_dim, shared_metric_embedding_dim=text_encoder.embedding_dim)
contrastive_learning = ContrastiveLearning()

In [ ]:
tfw = TrainingFramework(ecg_encoder, 
                        text_encoder, 
                        shared_metric_space, 
                        contrastive_learning, 
                        report_decoder, 
                        TRAINING_PARAMS)

In [ ]:
tfw.train(tr_dl, vl_dl)